In [ ]:
!pip install transformers datasets torch pandas matplotlib evaluate

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
import re
from tqdm.auto import tqdm

from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
from datasets import load_dataset
import evaluate


In [ ]:

MODEL_NAME = "deepset/roberta-base-squad2"  

PHASE1_JSON = "librispeech_local_1500.json"  
#output
OUTPUT_RESULTS = "phase3b_qa_results.json"
OUTPUT_EVALUATION = "phase3b_evaluation.json"

# create QA pipeline
qa_pipeline = pipeline(
    "question-answering",
    model=MODEL_NAME,
    device=-1  
)

squad = load_dataset("squad_v2")
val_data = squad['validation']
EVAL_SAMPLE_SIZE = None 
if EVAL_SAMPLE_SIZE:
    val_sample = val_data.select(range(min(EVAL_SAMPLE_SIZE, len(val_data))))
else:
    val_sample = val_data

In [ ]:
val_sample[0]

In [ ]:

predictions = []
references = []


for i, example in enumerate(tqdm(val_sample)):
    # extract question and context
    question = example['question']
    context = example['context']
    
    # predict answer
    try:
        result = qa_pipeline(
            question=question,
            context=context
        )
        
        predicted_answer = result['answer']
        confidence = result['score']
    except Exception as e:
        predicted_answer = ""
        confidence = 0.0
    
    # get true answer
    if example['answers']['text']:
        true_answers = example['answers']['text']
    else:
        true_answers = []
    
    # save prediction
    predictions.append({
        'id': example['id'],
        'question': question,
        'predicted_answer': predicted_answer,
        'true_answers': true_answers,
        'confidence': confidence
    })
    
    references.append({
        'id': example['id'],
        'answers': example['answers']
    })

print(f"{len(predictions)} samples processed")

In [ ]:

def normalize_answer(s):
    import string
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def compute_exact_match(prediction, ground_truths):
    """exact match"""
    prediction = normalize_answer(prediction)
    for truth in ground_truths:
        if normalize_answer(truth) == prediction:
            return 1
    return 0

def compute_f1(prediction, ground_truths):
    """f1 score"""
    prediction_tokens = normalize_answer(prediction).split()
    
    f1_scores = []
    for truth in ground_truths:
        truth_tokens = normalize_answer(truth).split()
        
        common = Counter(prediction_tokens) & Counter(truth_tokens)
        num_same = sum(common.values())
        
        if num_same == 0:
            f1_scores.append(0)
        else:
            precision = num_same / len(prediction_tokens) if prediction_tokens else 0
            recall = num_same / len(truth_tokens) if truth_tokens else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            f1_scores.append(f1)
    
    return max(f1_scores) if f1_scores else 0

# caculate
em_scores = []
f1_scores = []

for pred in predictions:
    predicted = pred['predicted_answer']
    ground_truths = pred['true_answers'] if pred['true_answers'] else [""]
    
    em = compute_exact_match(predicted, ground_truths)
    f1 = compute_f1(predicted, ground_truths)
    
    em_scores.append(em)
    f1_scores.append(f1)

avg_em = np.mean(em_scores) * 100
avg_f1 = np.mean(f1_scores) * 100


print(f"\nExact Match (EM): {avg_em:.2f}%")
print(f"F1 Score:         {avg_f1:.2f}%")


In [ ]:
errors = []
correct = []

for i, (pred, em) in enumerate(zip(predictions, em_scores)):
    if em == 0:
        errors.append((i, pred))
    else:
        correct.append((i, pred))

print(f"\ncorrect: {len(correct)}")
print(f"error: {len(errors)}")


In [ ]:
confidences = [pred['confidence'] for pred in predictions]

print(f"\nconfidence caculation:")
print(f"  average: {np.mean(confidences):.2f}")
print(f"  median: {np.median(confidences):.2f}")



In [ ]:

# laod phase1 result
with open(PHASE1_JSON, 'r', encoding='utf-8') as f:
    phase1_data = json.load(f)
    
transcriptions = [r['transcription'] for r in phase1_data['results'] if r['success']]
    
sample_questions = [
        "What is this text about?",
        "Who is mentioned in this text?",
        "When did this happen?",
        "Where does this take place?",
        "What was the main event?"
    ]
    
print(f"\ntest the QA in top 5..")
    
qa_examples = []
    
for i, text in enumerate(transcriptions[:5]):
        print(f"\ntranscribe {i+1}:")
        print(f"context: {text[:200]}...")
        print(f"\nQA:")
        
        for question in sample_questions:
            try:
                result = qa_pipeline(
                    question=question,
                    context=text
                )
                
                if result['score'] > 0.1:  # only show answers with confidence >0.1
                    print(f"  Q: {question}")
                    print(f"  A: {result['answer']} (confidence_level: {result['score']:.2f})")
                    
                    qa_examples.append({
                        'text_index': i,
                        'text_preview': text[:100],
                        'question': question,
                        'answer': result['answer'],
                        'confidence': result['score']
                    })
            except:
                pass
    
    
    

In [ ]:
def ask_question(context, question, show_context=True):
    """"
    Args:
        context
        question
        show_context
    
    Returns:
        dict: include answer and confidence
    """
    if show_context:
        print(context[:300] + "..." if len(context) > 300 else context)
    print(f"question: {question}")
  
    result = qa_pipeline(
        question=question,
        context=context
    )
    
    print(f"\nanswer: {result['answer']}")
    print(f"confidence: {result['score']:.2f}")
    
    if result['score'] < 0.1:
        print("low confidence")
    elif result['score'] < 0.5:
        print("medium confidence")
    else:
        print("✓ high confidence")
    
    return result


# one example test
if transcriptions:
    ask_question(
        context=transcriptions[0],
        question="What is the main course?"
    )

In [ ]:
# save results
results_output = {
    'metadata': {
        'model': MODEL_NAME,
        'eval_samples': len(predictions),
        'metrics': {
            'exact_match': float(avg_em),
            'f1_score': float(avg_f1)
        }
    },
    'predictions': predictions 
}

with open(OUTPUT_RESULTS, 'w', encoding='utf-8') as f:
    json.dump(results_output, f, ensure_ascii=False, indent=2)



# evaluation report
evaluation_output = {
    'model': MODEL_NAME,
    'dataset': 'SQuAD v2',
    'eval_samples': len(predictions),
    'metrics': {
        'exact_match': float(avg_em),
        'f1_score': float(avg_f1),
        'avg_confidence': float(np.mean(confidences))
    },
    'performance_breakdown': {
        'correct_predictions': len(correct),
        'incorrect_predictions': len(errors),
        'accuracy': float(len(correct) / len(predictions) * 100)
    },
    'confidence_stats': {
        'mean': float(np.mean(confidences)),
        'median': float(np.median(confidences)),
        'min': float(np.min(confidences)),
        'max': float(np.max(confidences))
    },
    'application_on_transcriptions': {
        'tested': len(qa_examples),
        'examples': qa_examples
    }
}

with open(OUTPUT_EVALUATION, 'w', encoding='utf-8') as f:
    json.dump(evaluation_output, f, ensure_ascii=False, indent=2)
